# 01 - Data Preparation
Loads `b-mc2/sql-create-context` from Hugging Face, subsamples 15,000 examples, and formats prompt/completion pairs for QLoRA SFT. Runs anywhere (no GPU needed) - Colab or local.

In [ ]:
!pip install -q datasets

In [ ]:
import sys
sys.path.insert(0, '..')
from src.data_prep import build_sft_dataset, train_val_split

examples = build_sft_dataset(n=15000, seed=42, out_path='../data/sql_create_context_15k.jsonl')
print(f'Total examples: {len(examples)}')
examples[0]

In [ ]:
train, val = train_val_split(examples, val_ratio=0.05, seed=42)
print(f'Train: {len(train)}  Val: {len(val)}')

import json
with open('../data/train.jsonl', 'w', encoding='utf-8') as f:
    for ex in train:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
with open('../data/val.jsonl', 'w', encoding='utf-8') as f:
    for ex in val:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

Sanity-check a handful of DDL/question/answer triples with the execution evaluator before spending GPU time on fine-tuning - this catches malformed rows in the raw dataset early.

In [ ]:
from src.execution_evaluator import verify
import random

sample = random.sample(examples, 20)
ok = 0
for ex in sample:
    outcome = verify(ex['context'], ex['answer'], ex['answer'])
    ok += int(outcome.syntax_valid)
print(f'{ok}/{len(sample)} gold queries executed successfully against synthetic dummy data')